# ML-09 — Validation and Research Claim Audit

In [3]:
!git clone https://github.com/Dilip-Git18/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 146, done.
remote: Counting objects: 100% (146/146), done.
remote: Compressing objects: 100% (98/98), done.
remote: Total 146 (delta 57), reused 105 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (146/146), 1.87 MiB | 6.50 MiB/s, done.
Resolving deltas: 100% (57/57), done.


In [4]:
!find . -name "*.csv"

./flyrank-ml-internship/outputs/refresh_queue_sample.csv
./flyrank-ml-internship/data/raw/content_refresh_anonymized.csv
./sample_data/mnist_train_small.csv
./sample_data/california_housing_train.csv
./sample_data/california_housing_test.csv
./sample_data/mnist_test.csv


In [5]:
import pandas as pd

df = pd.read_csv("./flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")

print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [6]:
df["is_declining_label"] = (
    df["trend_direction"].str.lower() == "down"
).astype(int)

print(df["is_declining_label"].value_counts())

is_declining_label
1    16262
0    13738
Name: count, dtype: int64


In [7]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'is_declining_label']


In [8]:
# ============================================
# 2. My model under an honest split (before/after)
# ============================================

import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    GroupKFold,
    cross_validate
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

TARGET = "is_declining_label"

# -----------------------------
# Remove leakage columns
# -----------------------------

DROP_COLUMNS = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    TARGET
]

X = df.drop(columns=DROP_COLUMNS)
y = df[TARGET]
groups = df["client_id"]

print("Dataset Shape :", df.shape)
print("Features :", X.shape[1])

# -----------------------------
# Feature Types
# -----------------------------

categorical = X.select_dtypes(include="object").columns.tolist()
numeric = X.select_dtypes(exclude="object").columns.tolist()

preprocessor = ColumnTransformer(
    [
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]),
            numeric
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical
        ),
    ]
)

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", model)
])

# ======================================
# RANDOM SPLIT
# ======================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

pipeline.fit(X_train, y_train)

pred = pipeline.predict(X_test)

random_accuracy = accuracy_score(y_test, pred)
random_f1 = f1_score(y_test, pred)

print("\nRandom Split Results")
print("--------------------")
print("Accuracy :", random_accuracy)
print("F1 Score :", random_f1)

# ======================================
# GROUPKFOLD
# ======================================

gkf = GroupKFold(n_splits=5)

scores = cross_validate(
    pipeline,
    X,
    y,
    groups=groups,
    cv=gkf,
    scoring=["accuracy", "f1"]
)

print("\nGrouped Validation")
print("--------------------")
print("Accuracy :", scores["test_accuracy"].mean())
print("F1 Score :", scores["test_f1"].mean())

# ======================================
# BASE RATE
# ======================================

baseline = y.value_counts(normalize=True).max()

print("\nMajority Baseline :", baseline)

comparison = pd.DataFrame({
    "Validation":[
        "Random Split",
        "Grouped Split",
        "Majority Baseline"
    ],
    "Accuracy":[
        random_accuracy,
        scores["test_accuracy"].mean(),
        baseline
    ],
    "F1":[
        random_f1,
        scores["test_f1"].mean(),
        np.nan
    ]
})

comparison

Dataset Shape : (30000, 45)
Features : 40

Random Split Results
--------------------
Accuracy : 0.8626666666666667
F1 Score : 0.8773079213817748

Grouped Validation
--------------------
Accuracy : 0.8335736046991601
F1 Score : 0.8504322940771099

Majority Baseline : 0.5420666666666667


,Validation,Accuracy,F1
0,Random Split,0.862667,0.877308
1,Grouped Split,0.833574,0.850432
2,Majority Baseline,0.542067,NaN


## 1. Two paper findings + my methodology questions

### Finding 1

The FlyRank research paper reports that AI-assisted SEO workflows can improve content production efficiency and help teams produce optimized content at scale.

My methodology question

How was "improvement" measured? Was the label based on objective SEO metrics (clicks, impressions, rankings, conversions), and was the same definition used consistently for every client? If different clients used different success criteria, the reported improvement could be difficult to compare.

### Finding 2

The paper suggests that AI-assisted optimization is associated with improved SEO performance.

My methodology question

Did the validation evaluate completely unseen clients, or were random train/test splits used? Because content from the same client often shares characteristics, a grouped validation using client_id would provide stronger evidence that the findings generalize to new clients rather than reflecting memorization.

The purpose of these questions is not to criticize the research paper but to understand how validation choices influence the strength of the conclusions. Asking these questions also helps improve the reliability of my own machine learning project.

## 2. My model under an honest split (before/after)

In Week 5, I evaluated my model using a standard random train/test split. For this assignment, I re-ran the model using a more honest validation strategy with **GroupKFold**, grouping by **client_id**. This prevents content from the same client appearing in both the training and testing sets, providing a better estimate of how the model performs on unseen clients.

### Results

| Validation Strategy | Accuracy | F1 Score |
|---------------------|---------:|---------:|
| Random Split | **0.8627** | **0.8773** |
| Grouped Split (GroupKFold) | **0.8336** | **0.8504** |
| Majority Class Baseline | **0.5421** | — |

### Interpretation

The grouped validation accuracy is lower than the random split accuracy. This suggests that the random split benefited from similarities between content belonging to the same client. The grouped split is a more realistic estimate of how the model would perform on new clients.

Even with the stricter validation strategy, the model performs substantially better than the majority-class baseline (54.2%), indicating that it has learned useful patterns from the available features rather than relying only on the most common class.

## 3. Leakage Audit

### Potential Leakage Sources

During this audit, I reviewed the final feature set to identify any columns that could unintentionally reveal the target label.

| Feature | Leakage Risk | Decision |
|---------|--------------|----------|
| trend_direction | High | Removed |
| trend_pct | High | Removed |
| client_id | Medium | Used only for GroupKFold splitting |
| content_id | Medium | Removed |
| Other content and performance features | Low | Retained |

### Why these features were removed

- **trend_direction** directly defines the target label (`is_declining_label`), so including it would cause perfect label leakage.
- **trend_pct** is used to calculate `trend_direction`, making it another label-derived feature.
- **client_id** is an identifier rather than a predictive feature. Instead of using it as input, I used it only to create grouped validation splits.
- **content_id** uniquely identifies each content item and therefore provides no meaningful predictive signal.

### Honest Validation

The grouped validation score (Accuracy = **0.8336**) is lower than the random split score (Accuracy = **0.8627**). This indicates that the random split likely benefited from client-specific similarities, while the grouped split provides a more realistic estimate of model performance on unseen clients.

Overall, I did not identify evidence of label leakage in the final feature set after removing the leakage-prone columns.

## 4. Claim Rewrite

### Original Claim

> The Random Forest model accurately predicts declining content.

### Revised Claim

Under grouped validation, the Random Forest model **observed** an accuracy of approximately **83.4%** and an F1-score of **85.0%** on this dataset.

These results represent **measured** performance for the available data and should be interpreted as **directional** evidence rather than proof that the model will generalize to every client or future dataset.

The model is intended as a **decision-support** tool to help prioritize content for review instead of making fully automated decisions.

# Self-check

- ✅ Every section above is completed with both markdown explanations and executable code.
- ✅ The notebook runs from top to bottom without errors.
- ✅ No client names, URLs, or private queries are included.
- ✅ My claims use careful language such as **observed**, **measured**, **directional**, and **decision-support**.
- ✅ The notebook is ready to be committed under `work/notebooks/w06_validation_audit.ipynb` and submitted with my public GitHub repository URL.